# TP3 sur Colab : l'interface de démonstration

Streamlit ne s'affiche pas directement dans Colab : le serveur tourne sur la machine virtuelle,
qui n'est pas joignable depuis votre navigateur. On passe donc par un **tunnel** qui lui donne
une adresse publique le temps de la session.

Mêmes étapes et mêmes contrôles que sur un poste local : S1 à S4.


## Étape A1. Récupérer le dépôt et installer (contrôle S1, première partie)


In [1]:
ORGANISATION = "anasmouquinee"   # votre compte ou organisation GitHub
DEPOT = "sentiment-app"

!git clone https://github.com/{ORGANISATION}/{DEPOT}.git
%cd {DEPOT}
!pip install -q streamlit pyarrow


Cloning into 'sentiment-app'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 135 (delta 48), reused 125 (delta 40), pack-reused 0
Receiving objects: 100% (135/135), 52.10 KiB | 2.45 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/sentiment-app


Si le kit TP3 n'est pas encore dans votre dépôt, téléversez `tp3-streamlit-kit.zip`
avec l'icône dossier du panneau de gauche, puis exécutez la cellule suivante.


In [2]:
import os
if not os.path.exists('ui/app.py'):
    !unzip -q -o ../tp3-streamlit-kit.zip -d ..
    !cd ../tp3-streamlit && python appliquer_tp3.py ../{DEPOT}
else:
    print('ui/app.py déjà présent')


ui/app.py déjà présent


## Étape A2. Entraîner le modèle

Le modèle n'est jamais versionné : il faut le reproduire ici, comme sur n'importe quelle machine neuve.
Deux à trois minutes avec le jeu Allociné.


In [3]:
!python src/train.py --allocine
!ls -lh models/


Source : allocine, 20000 avis (10204 négatifs)
16000 avis d'entraînement, 4000 de test
Vocabulaire : 149529 termes
F1 macro sur le jeu de test : 0.920 (seuil 0.85)
Modèle : models/sentiment.joblib
Métriques : models/metrics.json (versionné dans Git, contrairement au modèle)
total 3.8M
-rw-r--r-- 1 root root  192 Oct 12 17:02 metrics.json
-rw-r--r-- 1 root root 3.8M Oct 12 17:02 sentiment.joblib


## Étape A3. Lancer l'interface derrière un tunnel (contrôle S1)

La commande affiche d'abord un **mot de passe de tunnel** : c'est l'adresse IP publique de la machine.
Copiez-la, ouvrez le lien `loca.lt` qui s'affiche ensuite, et collez-la dans le champ demandé.


In [4]:
!npm install -q -g localtunnel 2>/dev/null
print('Mot de passe du tunnel (à coller sur la page loca.lt) :')
!curl -s https://loca.lt/mytunnelpassword


/tools/node/bin/lt -> /tools/node/lib/node_modules/localtunnel/bin/lt.js
+ localtunnel@2.0.2
added 22 packages from 22 contributors in 1.45s
Mot de passe du tunnel (à coller sur la page loca.lt) :
34.125.89.214


In [5]:
!streamlit run ui/app.py --server.headless true --server.port 8501 &>/content/streamlit.log &
import time; time.sleep(8)
!npx --yes localtunnel --port 8501


your url is: https://sweet-worms-shout.loca.lt


La cellule reste en cours d'exécution tant que le tunnel vit : c'est normal.
Pour l'arrêter, interrompez la cellule (bouton stop).

**Si le lien ne répond pas**, vérifiez le journal du serveur :


In [6]:
!tail -20 /content/streamlit.log


2026-10-12 17:33:53.079 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.89.214:8501


## Étapes B et C. Les deux onglets (contrôles S2 et S3)

Dans la page ouverte par le tunnel :

1. **Onglet « Un avis »** : saisissez un avis clairement positif, un clairement négatif,
   puis un mitigé (« service impeccable au début, mais finalement produit cassé »).
   **S2 :** l'avis mitigé doit afficher « à vérifier », pas un verdict tranché.
2. **Onglet « Un fichier CSV »** : téléversez `data/sample_reviews.csv` depuis votre poste
   (le fichier est aussi dans le dépôt cloné ici).
   **S3 :** trois métriques, un graphique, un tableau trié, un bouton de téléchargement.


## Étape D. Les tests (contrôle S4)

Les tests d'interface ne lancent pas Streamlit : ils importent `ui/helpers.py`.
Ils tournent donc ici sans tunnel ni serveur.


In [7]:
!pip install -q pytest
!python -m pytest -q --deselect tests/test_environment.py::test_virtualenv_active
# attendu : 44 passed (les 45 du TP moins test_virtualenv_active, sans objet sur Colab)


............................................                             [100%]
44 passed, 1 deselected in 1.55s


## Étape E. Pousser le travail

Colab efface `/content` à la fin de la session : **tout est perdu si vous ne poussez pas.**
Le modèle, lui, n'a pas à être conservé : il se réentraîne par une commande.


In [8]:
from getpass import getpass
JETON = getpass("Jeton d'accès personnel GitHub : ")

!git config user.name "Anas Mouqine" && git config user.email "anasmouquine2@gmail.com"
!git add -A && git commit -q -m "Semaine 3 : interface de demonstration Streamlit"
!git remote set-url origin https://{JETON}@github.com/{ORGANISATION}/{DEPOT}.git
!git push


Jeton d'accès personnel GitHub : ········
To https://github.com/anasmouquinee/sentiment-app.git
   7e7a510..1d38856  main -> main


## Les trois différences avec un poste local

**Le tunnel.** Sur votre portable, `streamlit run ui/app.py` suffit et le navigateur s'ouvre seul.
Ici, le serveur tourne sur une machine distante : sans tunnel, rien n'est joignable.
L'adresse `loca.lt` est publique et temporaire ; ne laissez pas tourner une interface
qui afficherait des données réelles.

**Pas d'environnement virtuel.** Colab installe dans son Python système :
`test_virtualenv_active` est écarté, d'où 44 tests au lieu de 45.

**Rien ne survit.** Poussez sur GitHub avant de fermer l'onglet.
